# YOLO CIFAR10 Image Source: Teacher → Pruned Student → Logits Distillation (Detection)

This notebook uses CIFAR10 images as an input source while running a YOLOv8n **detection** pruning + logits distillation flow:
1. Load YOLOv8n detection teacher from Ultralytics checkpoint.
2. Build a `MaseGraph` from a fresh detection model and run unstructured pruning to create the student.
3. Distill teacher logits into the pruned student using CIFAR10 mini-batches (images only).

In [1]:
det_model_checkpoint = "yolov8n.pt"
det_device = "cuda"
cifar_root = "./data"

image_size = 320
cifar_batch_size = 16
cifar_train_subset_size = 2048
cifar_val_subset_size = 512

cifar_prune_sparsity = 0.3
cifar_kd_alpha = 0.8
cifar_kd_temperature = 2.0
cifar_kd_steps = 200
lr = 1e-4
seed = 42

## Imports and setup

In [2]:
import copy
import random
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from ultralytics import YOLO

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/")

repo_root = find_repo_root(Path.cwd().resolve())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from chop import MaseGraph
import chop.passes as passes
from chop.models.yolo import get_yolo_detection_model

from mase_kd.core.losses import DistillationLossConfig
from mase_kd.vision.yolo_kd import YOLOLogitsDistiller

print(f"Repo root: {repo_root}")
print(f"Using src path: {src_path}")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.11/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


Repo root: /workspace/mase-kd
Using src path: /workspace/mase-kd/src


In [3]:
if det_device == "cuda" and not torch.cuda.is_available():
    det_device = "cpu"

torch.manual_seed(seed)
random.seed(seed)

print(f"Using device: {det_device}")

Using device: cuda


## CIFAR10 image dataset and dataloaders

In [4]:
cifar_mean = (0.4914, 0.4822, 0.4465)
cifar_std = (0.2470, 0.2435, 0.2616)

cifar_transform_train = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])
cifar_transform_eval = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

train_full = datasets.CIFAR10(root=cifar_root, train=True, transform=cifar_transform_train, download=True)
val_full = datasets.CIFAR10(root=cifar_root, train=False, transform=cifar_transform_eval, download=True)

subset_rng = torch.Generator().manual_seed(seed)
train_indices = torch.randperm(len(train_full), generator=subset_rng)[:cifar_train_subset_size]
val_indices = torch.randperm(len(val_full), generator=subset_rng)[:cifar_val_subset_size]

train_ds = Subset(train_full, train_indices.tolist())
val_ds = Subset(val_full, val_indices.tolist())

train_loader = DataLoader(train_ds, batch_size=cifar_batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cifar_batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train subset: {len(train_ds)} | Val subset: {len(val_ds)}")

Train subset: 2048 | Val subset: 512


## Load teacher model from Ultralytics (YOLOv8n detection checkpoint)

In [5]:
ultra_teacher = YOLO(det_model_checkpoint)
teacher_det_model = ultra_teacher.model.to(det_device)
teacher_det_model.eval()

print(f"Loaded teacher checkpoint: {det_model_checkpoint}")

Loaded teacher checkpoint: yolov8n.pt


## Build MASE graph and prune to create student

In [6]:
trace_input_det = torch.randn(1, 3, image_size, image_size)

student_seed_det_model = get_yolo_detection_model(det_model_checkpoint)
student_seed_det_model.train()

mg_det = MaseGraph(student_seed_det_model, cf_args={"x": trace_input_det})
mg_det, _ = passes.init_metadata_analysis_pass(mg_det)

placeholder_names_det = [
    node.name for node in mg_det.fx_graph.nodes if node.op == "placeholder"
 ]
dummy_in_det = {name: trace_input_det for name in placeholder_names_det}

mg_det, _ = passes.add_common_metadata_analysis_pass(
    mg_det,
    pass_args={
        "dummy_in": dummy_in_det,
        "add_value": True,
    },
)

pruning_config_det = {
    "weight": {
        "sparsity": cifar_prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
        "granularity": "elementwise",
    },
    "activation": {
        "sparsity": cifar_prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
        "granularity": "elementwise",
    },
}

mg_det, _ = passes.prune_transform_pass(mg_det, pass_args=pruning_config_det)
student_det_model = mg_det.model.to(det_device)

print(f"Metadata placeholders: {placeholder_names_det}")
print("Pruning complete; using pruned model as student.")


                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128

/usr/local/lib/python3.11/dist-packages/torch/fx/_symbolic_trace.py:906: UserWarning: Was not able to add assertion to guarantee correct input x to specialized function. It is up to the user to make sure that your inputs match the inputs you specialized the function with.
  warnings.warn(
INFO     Pruning module: model_0_conv
INFO     Pruning module: model_1_conv
INFO     Pruning module: model_2_cv1_conv
INFO     Pruning module: model_2_m_0_cv1_conv
INFO     Pruning module: model_2_m_0_cv2_conv
INFO     Pruning module: model_2_cv2_conv
INFO     Pruning module: model_3_conv
INFO     Pruning module: model_4_cv1_conv
INFO     Pruning module: model_4_m_0_cv1_conv
INFO     Pruning module: model_4_m_0_cv2_conv
INFO     Pruning module: model_4_m_1_cv1_conv
INFO     Pruning module: model_4_m_1_cv2_conv
INFO     Pruning module: model_4_cv2_conv
INFO     Pruning module: model_5_conv
INFO     Pruning module: model_6_cv1_conv
INFO     Pruning module: model_6_m_0_cv1_conv
INFO     Pruning module: m

Metadata placeholders: ['x_1']
Pruning complete; using pruned model as student.


## Distill teacher into pruned student on CIFAR10 images

In [7]:
kd_config_det = DistillationLossConfig(
    alpha=cifar_kd_alpha,
    temperature=cifar_kd_temperature,
)
distiller_det = YOLOLogitsDistiller(
    teacher=teacher_det_model,
    student=student_det_model,
    kd_config=kd_config_det,
    device=det_device,
)

pruned_no_kd_model = copy.deepcopy(student_det_model).to(det_device)
pruned_no_kd_model.eval()

optimizer = torch.optim.Adam(student_det_model.parameters(), lr=lr)
loss_history = []

train_iter = iter(train_loader)
for step in range(1, cifar_kd_steps + 1):
    try:
        images, _ = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        images, _ = next(train_iter)

    batch = {
        "images": images.to(det_device),
    }
    output = distiller_det.train_step(batch=batch, optimizer=optimizer)
    loss_history.append(output.total_loss)

    if step == 1 or step % 10 == 0 or step == cifar_kd_steps:
        print(
            f"Step {step:03d} | total={output.total_loss:.6f} | "
            f"hard={output.hard_loss:.6f} | soft={output.soft_loss:.6f}"
        )

Step 001 | total=1.011636 | hard=0.000000 | soft=1.264545
Step 010 | total=0.493991 | hard=0.000000 | soft=0.617489
Step 020 | total=0.341770 | hard=0.000000 | soft=0.427213
Step 030 | total=0.202983 | hard=0.000000 | soft=0.253729
Step 040 | total=0.166277 | hard=0.000000 | soft=0.207847
Step 050 | total=0.122691 | hard=0.000000 | soft=0.153364
Step 060 | total=0.115856 | hard=0.000000 | soft=0.144820
Step 070 | total=0.093548 | hard=0.000000 | soft=0.116935
Step 080 | total=0.090894 | hard=0.000000 | soft=0.113618
Step 090 | total=0.083889 | hard=0.000000 | soft=0.104862
Step 100 | total=0.082753 | hard=0.000000 | soft=0.103442
Step 110 | total=0.080156 | hard=0.000000 | soft=0.100195
Step 120 | total=0.072830 | hard=0.000000 | soft=0.091038
Step 130 | total=0.078895 | hard=0.000000 | soft=0.098619
Step 140 | total=0.069775 | hard=0.000000 | soft=0.087218
Step 150 | total=0.065923 | hard=0.000000 | soft=0.082404
Step 160 | total=0.068350 | hard=0.000000 | soft=0.085438
Step 170 | tot

## Validation performance (teacher vs student)

In [8]:
import time

def _flatten_for_eval(output):
    if isinstance(output, torch.Tensor):
        if output.ndim == 0:
            return output.reshape(1, 1)
        if output.shape[0] == 0:
            return output.reshape(1, 0)
        return output.reshape(output.shape[0], -1)

    if isinstance(output, dict):
        parts = []
        for key in sorted(output.keys()):
            value = output[key]
            if isinstance(value, (torch.Tensor, dict, list, tuple)):
                part = _flatten_for_eval(value)
                if part is not None:
                    parts.append(part)
        if not parts:
            return None
        batch0 = parts[0].shape[0]
        if all(part.shape[0] == batch0 for part in parts):
            return torch.cat(parts, dim=1)
        parts = [part.mean(dim=0, keepdim=True) for part in parts]
        width = min(part.shape[1] for part in parts)
        if width == 0:
            return None
        return torch.cat([part[:, :width] for part in parts], dim=0)

    if isinstance(output, (list, tuple)):
        parts = []
        for item in output:
            if isinstance(item, (torch.Tensor, dict, list, tuple)):
                part = _flatten_for_eval(item)
                if part is not None:
                    parts.append(part)
        if not parts:
            return None
        batch0 = parts[0].shape[0]
        if all(part.shape[0] == batch0 for part in parts):
            return torch.cat(parts, dim=1)
        parts = [part.mean(dim=0, keepdim=True) for part in parts]
        width = min(part.shape[1] for part in parts)
        if width == 0:
            return None
        return torch.cat([part[:, :width] for part in parts], dim=0)

    return None

def _align_logits_for_kd(student_logits, teacher_logits):
    if student_logits.shape[0] != teacher_logits.shape[0]:
        student_logits = student_logits.mean(dim=0, keepdim=True)
        teacher_logits = teacher_logits.mean(dim=0, keepdim=True)

    width = min(student_logits.shape[1], teacher_logits.shape[1])
    if width == 0:
        return None, None

    return student_logits[:, :width], teacher_logits[:, :width]

@torch.no_grad()
def evaluate_model_on_cifar10_val(model, loader, device, max_batches=8):
    model.eval()
    batches = 0
    samples = 0
    sum_abs_logit = 0.0
    sum_sq_logit = 0.0
    total_forward_ms = 0.0

    for batch_idx, (images, _) in enumerate(loader):
        if batch_idx >= max_batches:
            break

        images = images.to(device)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        outputs = model(images)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        flat = _flatten_for_eval(outputs)
        if flat is None or flat.numel() == 0:
            continue

        total_forward_ms += (t1 - t0) * 1000.0
        sum_abs_logit += flat.abs().mean().item()
        sum_sq_logit += flat.pow(2).mean().item()
        batches += 1
        samples += images.shape[0]

    return {
        "batches": batches,
        "samples": samples,
        "avg_forward_ms_per_batch": total_forward_ms / max(batches, 1),
        "avg_abs_logit": sum_abs_logit / max(batches, 1),
        "avg_logit_l2": (sum_sq_logit / max(batches, 1)) ** 0.5,
    }

@torch.no_grad()
def evaluate_validation_losses(teacher, student, loader, device, temperature=2.0, max_batches=8):
    teacher.eval()
    student.eval()
    total_teacher_loss = 0.0
    total_student_loss = 0.0
    total_kd_loss = 0.0
    used_batches = 0

    for batch_idx, (images, _) in enumerate(loader):
        if batch_idx >= max_batches:
            break

        images = images.to(device)
        teacher_outputs = teacher(images)
        student_outputs = student(images)

        teacher_logits = _flatten_for_eval(teacher_outputs)
        student_logits = _flatten_for_eval(student_outputs)
        if teacher_logits is None or student_logits is None:
            continue

        student_logits, teacher_logits = _align_logits_for_kd(student_logits, teacher_logits)
        if student_logits is None or teacher_logits is None:
            continue

        pseudo_targets = teacher_logits.argmax(dim=1)

        teacher_loss = torch.nn.functional.cross_entropy(teacher_logits, pseudo_targets)
        student_loss = torch.nn.functional.cross_entropy(student_logits, pseudo_targets)

        student_log_probs = torch.nn.functional.log_softmax(student_logits / temperature, dim=1)
        teacher_probs = torch.nn.functional.softmax(teacher_logits / temperature, dim=1)
        kd_loss = torch.nn.functional.kl_div(
            student_log_probs, teacher_probs, reduction="batchmean"
        ) * (temperature ** 2)

        total_teacher_loss += teacher_loss.item()
        total_student_loss += student_loss.item()
        total_kd_loss += kd_loss.item()
        used_batches += 1

    return (
        total_teacher_loss / max(used_batches, 1),
        total_student_loss / max(used_batches, 1),
        total_kd_loss / max(used_batches, 1),
        used_batches,
    )

if "loss_history" in globals() and len(loss_history) > 0:
    print(f"Recorded {len(loss_history)} KD steps")
    print(f"First loss: {loss_history[0]:.6f}")
    print(f"Last loss:  {loss_history[-1]:.6f}")
else:
    print("No training loss history found in current kernel session.")

teacher_metrics = evaluate_model_on_cifar10_val(
    teacher_det_model, val_loader, det_device, max_batches=8
)
student_metrics = evaluate_model_on_cifar10_val(
    student_det_model, val_loader, det_device, max_batches=8
)
teacher_val_loss, student_val_loss, val_kd_loss, loss_batches = evaluate_validation_losses(
    teacher_det_model, student_det_model, val_loader, det_device,
    temperature=cifar_kd_temperature, max_batches=8
)

pruned_no_kd_val_loss = None
if "pruned_no_kd_model" in globals():
    _, pruned_no_kd_val_loss, _, _ = evaluate_validation_losses(
        teacher_det_model, pruned_no_kd_model, val_loader, det_device,
        temperature=cifar_kd_temperature, max_batches=8
)
else:
    print("No no-KD pruned model found; run Cell 14 first to create `pruned_no_kd_model`.")

print("Teacher validation performance (CIFAR10):")
print(
    f"  batches={teacher_metrics['batches']} | samples={teacher_metrics['samples']} | "
    f"avg_forward_ms/batch={teacher_metrics['avg_forward_ms_per_batch']:.3f} | "
    f"avg_abs_logit={teacher_metrics['avg_abs_logit']:.6f} | "
    f"avg_logit_l2={teacher_metrics['avg_logit_l2']:.6f}"
)

print("Pruned student validation performance (CIFAR10):")
print(
    f"  batches={student_metrics['batches']} | samples={student_metrics['samples']} | "
    f"avg_forward_ms/batch={student_metrics['avg_forward_ms_per_batch']:.3f} | "
    f"avg_abs_logit={student_metrics['avg_abs_logit']:.6f} | "
    f"avg_logit_l2={student_metrics['avg_logit_l2']:.6f}"
)

print(f"Teacher validation loss: {teacher_val_loss:.6f}")
print(f"Student validation loss (with KD): {student_val_loss:.6f}")
if pruned_no_kd_val_loss is not None:
    print(f"Pruned model validation loss (without KD): {pruned_no_kd_val_loss:.6f}")
print(
    f"Validation KD loss (teacher vs student, T={cifar_kd_temperature}): "
    f"{val_kd_loss:.6f} over {loss_batches} batches"
)

Recorded 200 KD steps
First loss: 1.011636
Last loss:  0.059038
Teacher validation performance (CIFAR10):
  batches=8 | samples=128 | avg_forward_ms/batch=38.123 | avg_abs_logit=8.073622 | avg_logit_l2=21.431927
Pruned student validation performance (CIFAR10):
  batches=8 | samples=128 | avg_forward_ms/batch=43.849 | avg_abs_logit=9.539653 | avg_logit_l2=11.814709
Teacher validation loss: 2.706845
Student validation loss (with KD): 9.777912
Pruned model validation loss (without KD): 9.869653
Validation KD loss (teacher vs student, T=2.0): 25.665023 over 8 batches


This notebook now runs a YOLOv8n **detection** teacher→student KD smoke flow using CIFAR10 as an image-only source. For task-faithful detection training, replace CIFAR10 with a detection dataset and include a detection hard loss (boxes + classes + DFL) alongside logits KD.